# 03 · Data Ingestion — Central Banks / Trade Agencies

**Goal:** pull country-level import series directly from official monetary/trade authorities,
where they offer a real, queryable API — as a complement to the aggregated Comtrade data.

**Fix applied in this version:** the previous version used BCRP series code `RD38112BM`,
which is the **Tumbes** customs office (a minor northern border crossing) — not a meaningful
series for this analysis. The correct code for Peru's main import gateway is `RD38118BM`
(**Marítima Del Callao**, which handles the large majority of Peru's seaborne trade,
including vehicles). This was confirmed against BCRP's own published series catalog.

**Known limitation:** BCRP's monthly customs series only run through **December 2023** at
the time of writing — they have not been updated with 2024–2025 data. This is a limitation
of the source itself, not of this notebook. Cross-check the "Última Actualización" column at
https://estadisticas.bcrp.gob.pe/estadisticas/series/mensuales/importaciones-por-aduana-m
before relying on this series for recent-year conclusions.

**Countries without a usable API:** Argentina (ADEFA), Ecuador (AEADE), and Bolivia do not
publish machine-readable trade/EV-adoption data — only press releases and PDF bulletins. This
was investigated directly (not assumed) and is documented with sourced figures in
`docs/data_sources.md`, to be used as qualitative context rather than pipeline data.

In [2]:
import sys
sys.path.append('../src')
from utils import call_json_api, save_processed, DATA_RAW
import pandas as pd
import requests

## 1. Peru — BCRP (Banco Central de Reserva del Perú)

API format:
```
https://estadisticas.bcrp.gob.pe/estadisticas/series/api/[series_codes]/[format]/[start]/[end]/[language]
```
Series used: `RD38118BM` = Marítima Del Callao, Peru's main seaport for imports (corrected —
see the note above). Browse more series at:
https://estadisticas.bcrp.gob.pe/estadisticas/series/mensuales/importaciones-por-aduana-m

In [3]:
SERIES_CODE = "RD38118BM"  # Marítima Del Callao — corrected from the Tumbes code used previously
FORMAT = "json"
PERIOD_START = "2019-1"
PERIOD_END = "2025-12"

url_bcrp = f"https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{SERIES_CODE}/{FORMAT}/{PERIOD_START}/{PERIOD_END}/eng"

try:
    data_bcrp = call_json_api(url_bcrp, retries=2)
    print("Connected OK. Response keys:", list(data_bcrp.keys()))
except Exception as e:
    print(f"Could not connect: {e}")
    data_bcrp = None

14:57:55 | WARNING | Attempt 1 on https://estadisticas.bcrp.gob.pe/estadisticas/series/api/RD38118BM/json/2019-1/2025-12/eng failed: Expecting value: line 1 column 1 (char 0)
14:57:57 | WARNING | Attempt 2 on https://estadisticas.bcrp.gob.pe/estadisticas/series/api/RD38118BM/json/2019-1/2025-12/eng failed: Expecting value: line 1 column 1 (char 0)


Could not connect: Could not fetch JSON from https://estadisticas.bcrp.gob.pe/estadisticas/series/api/RD38118BM/json/2019-1/2025-12/eng after 2 attempts: Expecting value: line 1 column 1 (char 0)
If the error is 403 / host blocked, check your internet connection or a corporate firewall/antivirus filtering the domain. Try pasting the URL directly into a browser first.


In [4]:
def parse_bcrp_response(data: dict) -> pd.DataFrame:
    """Convert BCRPData's JSON response into a tidy DataFrame."""
    if data is None:
        return pd.DataFrame()
    periods = data.get('periods', [])
    series_names = [s.get('name') for s in data.get('config', {}).get('series', [])]
    rows = []
    for p in periods:
        row = {'period': p.get('name')}
        for name, value in zip(series_names, p.get('values', [])):
            row[name] = value
        rows.append(row)
    return pd.DataFrame(rows)

df_bcrp = parse_bcrp_response(data_bcrp)

if not df_bcrp.empty:
    last_period_with_data = df_bcrp.dropna(subset=[c for c in df_bcrp.columns if c != 'period']).iloc[-1]['period']
    print(f"Last period with non-null data: {last_period_with_data}")
    print("If this is earlier than you expect, the source has not published more recent months yet.")

df_bcrp.head()

""


In [5]:
if not df_bcrp.empty:
    save_processed(df_bcrp, "bcrp_callao_imports_peru.csv")

## 2. Brazil — ComexStat (MDIC)

A real, free, public REST API — no registration required. Lets you filter by NCM code
(Brazil's HS-code equivalent) and country of origin, which is directly useful for the
China-vs-rest comparison.

Docs: https://comexstat.mdic.gov.br/en/home (see the "API" section)

**Note on the NCM code below:** `87038000` targets BEVs, but Brazil's NCM classification can
differ from the standard HS code in the last digit or two. Confirm the exact NCM via the
NCM search tool on the ComexStat site before treating this as authoritative for a report.

In [6]:
url_comexstat = "https://api-comexstat.mdic.gov.br/general"

payload_comexstat = {
    "flow": "import",
    "monthDetail": False,
    "period": {"from": "2021-01", "to": "2025-12"},
    "filters": [
        {"filter": "ncm", "values": ["87038000"]}  # BEV — verify exact NCM code before final use
    ],
    "details": ["country"],
    "metrics": ["metricFOB", "metricKG"]
}

try:
    resp = requests.post(url_comexstat, json=payload_comexstat, timeout=30)
    resp.raise_for_status()
    data_comexstat = resp.json()
    print("Connected OK to ComexStat.")
except Exception as e:
    print(f"Adjust the payload per the official docs if this fails: {e}")
    data_comexstat = None

Connected OK to ComexStat.


In [7]:
if data_comexstat:
    df_brazil = pd.json_normalize(data_comexstat.get('data', {}).get('list', []))
    display(df_brazil.head())
    if not df_brazil.empty:
        save_processed(df_brazil, "comexstat_ev_imports_brazil.csv")
else:
    print("No Brazil data this run — check the payload against ComexStat's official docs.")

,year,country,metricFOB,metricKG
0,2025,China,851489183,87300028
1,2025,Alemanha,53023113,2480407
2,2025,Bélgica,35923960,2586607
3,2025,Estados Unidos,28181861,1198847
4,2025,México,14368871,782525


14:58:40 | INFO | Saved -> data/processed/comexstat_ev_imports_brazil.csv (71 rows, 4 cols)


## 3. Argentina, Ecuador, Bolivia — no usable API

Investigated directly:

- **Argentina (ADEFA)** — publishes PDF annual reports and press bulletins only.
- **Ecuador (AEADE)** — the richest of the three in terms of detail (annual EV sales by
  brand, showing BYD's rapid rise), but distributed as PDF/press bulletins, not an API.
- **Bolivia** — no equivalent trade association with public bulletins was found.

Sourced figures for all three are documented in `docs/data_sources.md` and are meant to be
cited as qualitative context in the final report — not loaded here as pipeline data, to avoid
mixing API-verifiable numbers with manually transcribed ones inside the same automated table.

In [8]:
def download_pending_source(country: str, url: str, output_filename: str):
    """
    Template for once you confirm a real, queryable endpoint for a pending country.
    Example once you have a real URL:
        download_pending_source('Argentina', 'https://...', 'argentina_imports.csv')
    """
    from utils import download_csv, DATA_RAW
    return download_csv(url, DATA_RAW / output_filename)

print("Template ready — wire in a real call once a genuine endpoint is confirmed.")

Template ready — wire in a real call once a genuine endpoint is confirmed.


## Findings

These two series are complementary, not central to the main narrative: Brazil's ComexStat
data confirms country-of-origin detail consistent with the Comtrade findings in notebook 02,
and Peru's BCRP series provides a single-country cross-check for the region's largest Pacific
port.

**Limitation to keep in mind:** BCRP's published series currently end in December 2023, so
this source cannot speak to 2024-2025 trends — treat it as historical confirmation, not a
current-state indicator. Re-run this notebook periodically to check whether more recent
months have been published.